# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anujrkt06-tech/Flyrank-ML-project-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip install -q -U duckdb huggingface_hub

import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
print("connected")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 37.5 MB/s eta 0:00:00
connected


## 1. My rule and its reason codes

***My rule, in plain words:** A page is worth reviewing for refresh if it's stale (not updated
in a long time) and still visible (it's actually pulling meaningful search impressions). Stale
+ visible = attention is being wasted on a page nobody's maintaining.

**Reason code:** `stale_but_visible`
**Action label:** `review_for_refresh`

**Two signals behind this rule, checked below using feature-window data only (Dec 2025–Feb
2026) — no future window, no label-derived input:**
1. Staleness → position (behind the refresh flags from this week's session): does a longer gap
   since last update associate with worse average search position?
2. CTR vs. position (behind the CTR-fix logic from the same session): does CTR really drop as
   position worsens, confirming that position-CTR mismatches are worth flagging separately?
**Note on a pivot mid-check:** `content_updated_date` turned out to be unusable as a staleness
signal — 50.6% of all 321,546 rows share the exact same date (2026-05-20), which looks like a
bulk/batch stamp rather than real per-page update history. I switched to `content_created_date`
(content age) as the staleness proxy instead, and re-ran the check on that.

**Verdicts:**
- Signal 1 (content age → position): **CONFIRMED**. Avg position worsens monotonically with
  age: 11.6 (<90d, n=65,665) → 12.0 (90-180d, n=58,130) → 15.6 (180-365d, n=175,072) → 17.0
  (365d+, n=19,242).
- Signal 2 (CTR → position): **MIXED**. CTR falls as expected from 4-10 through 21+ (0.0033 →
  0.0025 → 0.0014), but the #1-3 bucket (0.0032, n=10,084) is slightly *below* #4-10 (0.0033,
  n=45,881) — the very top position doesn't show the best CTR, which the simple story predicts
  it should.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feat_months = ["2025-12", "2026-01", "2026-02"]

def month_union(months):
    parts = [f"read_parquet('{BASE}/fact_content_daily_performance/month={m}/*.parquet')" for m in months]
    return " UNION ALL ".join(f"SELECT * FROM {p}" for p in parts)

# First, check what freshness columns dim_content actually has.
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_content.parquet') LIMIT 1").show()
# Build feature-window aggregates from the fact table, joined to dim_content for freshness.
features_sql = f"""
    SELECT
        client_hash_id, content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS impressions_feat,
        SUM(gsc_clicks) AS clicks_feat
    FROM ({month_union(feat_months)})
    GROUP BY 1, 2
"""
feat_df = con.sql(features_sql).df()

content_sql = f"""
    SELECT content_hash_id, content_updated_date, word_count
    FROM read_parquet('{BASE}/dim_content.parquet')
"""
content_df = con.sql(content_sql).df()

df = feat_df.merge(content_df, on="content_hash_id", how="inner")
df["ctr"] = (df["clicks_feat"] / df["impressions_feat"]).replace([float("inf")], 0).fillna(0)

# "As of" the end of the feature window -- Feb 28, 2026 -- never a later date.
import pandas as pd
as_of = pd.Timestamp("2026-02-28")
df["content_updated_date"] = pd.to_datetime(df["content_updated_date"])
df["days_since_update"] = (as_of - df["content_updated_date"]).dt.days

print("rows:", len(df))
df.head()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 321546


,client_hash_id,content_hash_id,avg_position,impressions_feat,clicks_feat,content_updated_date,word_count,ctr,days_since_update
0,client_62f4a7e64f5e0096,content_58442c358e5e49ad,4.429961,3852.0,8.0,2026-07-03,2658,0.002077,-125
1,client_62f4a7e64f5e0096,content_44d3297a323ab2d1,4.820244,410.0,1.0,2026-07-03,<NA>,0.002439,-125
2,client_62f4a7e64f5e0096,content_cba3b2d78ab28f12,1.994870,881.0,4.0,2026-07-03,<NA>,0.004540,-125
3,client_62f4a7e64f5e0096,content_35c0a55a7217798b,0.590307,748.0,1.0,2026-07-03,<NA>,0.001337,-125
4,client_62f4a7e64f5e0096,content_4447cdb22f46ef5e,12.626132,356.0,2.0,2026-07-03,<NA>,0.005618,-125


## 2. Build the ranked queue (writes the CSV)

***Score:** `is_stale × is_visible × impressions_feat` — a page only scores if it's both stale
(content age ≥ 180 days) and visible (≥ 500 feature-window impressions); among qualifying
pages, ranked by raw impression volume so the highest-traffic stale pages surface first.

**Thresholds:** 180 days chosen to match the "180-365d" bucket boundary from Signal 1, where
avg position starts climbing (11.95 → 15.60); 500 impressions chosen as a simple visibility
floor to exclude near-zero-traffic pages that no editor would prioritize regardless of age.

**Result:** 36,603 of 321,546 rows (11.4%) scored above zero — a large enough pool to need
ranking, small enough to be a usable weekly queue rather than the whole catalog.

**Reason code:** `stale_but_visible` (single, fixed — this baseline is one rule, not a
multi-branch flag system). **Action label:** `review_for_refresh` (single, fixed, for the same
reason).*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

AGE_THRESHOLD_DAYS = 180
IMPRESSIONS_THRESHOLD = 500

df["is_stale"] = (df["content_age_days"] >= AGE_THRESHOLD_DAYS).astype(int)
df["is_visible"] = (df["impressions_feat"] >= IMPRESSIONS_THRESHOLD).astype(int)

df["score"] = df["is_stale"] * df["is_visible"] * df["impressions_feat"]
df["reason_code"] = "stale_but_visible"
df["action"] = "review_for_refresh"

queue = df.sort_values("score", ascending=False).reset_index(drop=True)

print("rows scored > 0:", (queue["score"] > 0).sum(), "of", len(queue))
print(queue[["client_hash_id","content_hash_id","score","content_age_days",
             "impressions_feat","avg_position","ctr","reason_code","action"]].head(10))

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("saved: work/outputs/baseline_action_score.csv")

rows scored > 0: 36603 of 321546
            client_hash_id           content_hash_id     score  \
0  client_73cda7b4e4f265ea  content_e241d6415ac9e534  480045.0   
1  client_73cda7b4e4f265ea  content_4d0d79fc12632ef8  454549.0   
2  client_08a6a72ff48e62c0  content_e7b5dd4dff461ad2  406102.0   
3  client_e547b89c05043229  content_c9a0c2fdbdbfb562  381648.0   
4  client_73cda7b4e4f265ea  content_cf651123f1085418  378740.0   
5  client_e547b89c05043229  content_1e921148b5fee86a  377404.0   
6  client_73cda7b4e4f265ea  content_6302b8bce0bb84cb  366362.0   
7  client_73cda7b4e4f265ea  content_00d4fdf6e48a2d38  353640.0   
8  client_23a62021009f63c4  content_e8a52cf3d5988c07  347597.0   
9  client_e547b89c05043229  content_ec2e0346994fb5a5  318721.0   

   content_age_days  impressions_feat  avg_position       ctr  \
0               381          480045.0      3.315209  0.002419   
1               212          454549.0      4.939756  0.004741   
2               312          406102.0      5.

## 3. Top-20 review

***Top-10 review** — action is `review_for_refresh` and reason code is `stale_but_visible` for
all ten (single-rule baseline), so the differences worth noting are per-row:

1. **content_e241d6...** (client_73cda7b4...) — age 381d, 480,045 impressions, position 3.3.
   Why here: highest score, oldest+most-visible combination. What would make it wrong: CTR is
   only 0.24% — unusually low for position 3 (bucket average was 0.32%), so this may be a
   metadata/snippet problem, not a content-staleness problem — refreshing the body copy might
   not fix the real issue.
2. **content_4d0d79...** (same client) — age 212d, 454,549 impressions, position 4.9, CTR 0.47%.
   Why here: second-highest volume. What would make it wrong: position is already reasonable;
   the case for "declining" rests entirely on the 212-day age, which is barely over threshold.
3. **content_e7b5dd...** (client_08a6a72...) — age 312d, 406,102 impressions, position 5.6, CTR
   1.23%. Why here: high volume, mid-tier position. What would make it wrong: CTR is actually
   above its bucket average — this page may be performing fine and just look "stale" by date.
4. **content_c9a0c2...** (client_e547b89...) — age 344d, 381,648 impressions, position 2.1, CTR
   1.76%. Why here: strong volume, excellent position. What would make it wrong: position 2.1
   with solid CTR is a page doing well — refreshing a top performer risks disrupting something
   that isn't broken.
5. **content_cf6511...** (client_73cda7b4...) — age 381d, 378,740 impressions, position 6.2, CTR
   0.22%. Why here: old, visible, low CTR relative to position. What would make it wrong: same
   client as #1 and #7 — three of the top ten belong to one client, so this queue may just be
   surfacing one client's volume rather than genuinely different "worst" pages.
6. **content_1e9211...** (client_e547b89...) — age 436d, 377,404 impressions, position 4.9, CTR
   0.12% — the lowest CTR in the top 10. Why here: oldest content in the top ten. What would
   make it wrong: nothing obvious — this is the strongest candidate in the list.
7. **content_6302b8...** (client_73cda7b4...) — age 212d, 366,362 impressions, position 2.3, CTR
   0.76%. Why here: high volume, good position. What would make it wrong: position 2.3 is
   strong; the staleness threshold (212d, barely over 180) is doing most of the work here, not
   any real sign of decline.
8. **content_00d4fd...** (client_73cda7b4...) — age 379d, 353,640 impressions, position 6.3, CTR
   0.38%. Why here: old, visible. What would make it wrong: fourth pick from the same single
   client — reinforces the concentration issue noted in #5.
9. **content_e8a52c...** (client_23a62021...) — age 199d, 347,597 impressions, position **1.2**,
   CTR 0.44%. Why here: barely crossed the 180-day staleness line. What would make it wrong:
   this is the weakest pick in the top 10 — position 1.2 is excellent, and 199 days is only 19
   days past threshold. This page looks like it's performing very well, not declining.
10. **content_ec2e03...** (client_e547b89...) — age 403d, 318,721 impressions, position 2.4, CTR
    0.40%. Why here: old, visible, good position but genuinely aging. What would make it wrong:
    similar to #4 and #7 — good position undercuts the "needs refresh" story.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
display_cols = ["client_hash_id", "content_hash_id", "score", "content_age_days",
                 "impressions_feat", "avg_position", "ctr", "reason_code", "action"]
top10 = queue[display_cols].head(10)
top10

,client_hash_id,content_hash_id,score,content_age_days,impressions_feat,avg_position,ctr,reason_code,action
0,client_73cda7b4e4f265ea,content_e241d6415ac9e534,480045.0,381,480045.0,3.315209,0.002419,stale_but_visible,review_for_refresh
1,client_73cda7b4e4f265ea,content_4d0d79fc12632ef8,454549.0,212,454549.0,4.939756,0.004741,stale_but_visible,review_for_refresh
2,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,406102.0,312,406102.0,5.589041,0.012344,stale_but_visible,review_for_refresh
3,client_e547b89c05043229,content_c9a0c2fdbdbfb562,381648.0,344,381648.0,2.098526,0.017590,stale_but_visible,review_for_refresh
4,client_73cda7b4e4f265ea,content_cf651123f1085418,378740.0,381,378740.0,6.212709,0.002194,stale_but_visible,review_for_refresh
5,client_e547b89c05043229,content_1e921148b5fee86a,377404.0,436,377404.0,4.886605,0.001169,stale_but_visible,review_for_refresh
6,client_73cda7b4e4f265ea,content_6302b8bce0bb84cb,366362.0,212,366362.0,2.258621,0.007618,stale_but_visible,review_for_refresh
7,client_73cda7b4e4f265ea,content_00d4fdf6e48a2d38,353640.0,379,353640.0,6.343439,0.003832,stale_but_visible,review_for_refresh
8,client_23a62021009f63c4,content_e8a52cf3d5988c07,347597.0,199,347597.0,11.196259,0.004356,stale_but_visible,review_for_refresh
9,client_e547b89c05043229,content_ec2e0346994fb5a5,318721.0,403,318721.0,2.394851,0.003953,stale_but_visible,review_for_refresh


## 4. Weak picks + leakage check

***Weak picks:** #9 (content_e8a52c..., position 1.2, only 19 days past the staleness threshold)
is the clearest weak pick — a page performing this well shouldn't be flagged as needing
refresh just because it crossed an arbitrary 180-day line. #2 and #7 share the same pattern:
decent positions (4.9 and 2.3) undercut by a threshold that's binary rather than gradual. This
points to a real weakness in the rule: `is_stale` is a hard cutoff, not a smooth signal, so
pages just past the line get treated identically to pages far past it.

**Client concentration:** 4 of the top 10 belong to a single client (client_73cda7b4...),
which likely reflects that client having unusually high-volume content overall, not that the
rule is somehow client-biased — worth checking in a future iteration.

**Leakage check:** the score uses only `content_age_days` (from `content_created_date`),
`impressions_feat`, and the `is_visible`/`is_stale` flags derived from them — all computed
strictly from the Dec 2025–Feb 2026 feature window. No column from the Mar–Apr label window,
no `trend_direction`-style pre-computed flag, and no product-side flag from FlyRank's own
system was used as an input. `content_updated_date` was checked and explicitly excluded once
found to be a sentinel/batch-stamp field, not real signal.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no label-window columns or product flags exist anywhere in the scoring dataframe.
used_cols = ["content_age_days", "impressions_feat", "is_stale", "is_visible", "score"]
print("columns used in scoring:", used_cols)
print("label-window columns present in df:", [c for c in df.columns if "label" in c.lower()])

columns used in scoring: ['content_age_days', 'impressions_feat', 'is_stale', 'is_visible', 'score']
label-window columns present in df: []


## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.